# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/daniyalhaider236/flyrank-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## 1. Signal checks + my rule and reason codes

### Signal check 1 — staleness (behind FlyRank's refresh flags)

**Hypothesis a refresh flag leans on:** the longer a page has gone untouched, the more likely it is to
be a page that's currently declining and worth reviewing.

I bucket `days_since_last_update` and compare the decline rate (`trend_direction == "down"`) in each
bucket against the base rate. Table + n are in the code cell below.

**Verdict: MIXED.** Decline rate does rise above the 54.2% base rate through the 91–180 day bucket
(61.1%, n=9,171) — that part supports the refresh-flag assumption. But it is not monotonic: the 181+
bucket drops back to 47.1% (n=174, a small sample, so treat it cautiously rather than as a firm reversal).
A plain "older = more likely declining" story doesn't hold cleanly at the tail, so staleness alone is not
a safe standalone predictor — it needs a second signal alongside it.

### Signal check 2 — visibility / volume (behind FlyRank's quick-win logic)

**Hypothesis quick-win logic leans on:** pages with more search visibility matter more, because a fix on
a high-volume page has a bigger payoff — so volume should track review priority.

I bucket by `impression_tier` (the dataset's own transparent volume tiers) and compare decline rate per
tier. Table + n are below.

**Verdict: MIXED.** Decline rate is *not* monotonic with volume — it rises from `low` (45.4%) to
`moderate` (61.5%) and `good` (58.6%), then falls back to 46.2% at `excellent`. So volume does not predict
*whether* a page is declining on its own. What this tells me for the rule: don't use volume as a decline
predictor — use it as an **impact multiplier**. Among pages that are already flagged stale, the ones with
more visibility are worth reviewing first because fixing them matters more, not because high volume itself
signals a problem. This is a negative result that shaped the rule for the better: it's the reason the score
below multiplies by raw impressions instead of trying to use volume as a threshold condition on its own.

### The rule, in plain words

*A page is worth reviewing first if it (1) hasn't been touched in 90+ days AND (2) still pulls in real
search visibility (300+ impressions in the last 90 days). Among pages that clear both bars, the ones with
more impressions are ranked higher, because a fix there reaches more people.*

### Score, reason code, action label

- **Score** = `stale × visible × impressions_90d` (0 for anything that doesn't clear both bars; otherwise
  scaled by audience size)
- **Reason code** (ONE code): `"stale_visible_page"` when both conditions hold, else `"not_flagged"`
- **Action label**: `"review_for_refresh"` when flagged, else `"monitor"`

Thresholds: `days_since_last_update >= 90` (chosen because that's where the staleness signal check above
showed the real lift) and `impressions_90d >= 300` (the dataset's own "moderate" visibility cutoff).

In [9]:

import pandas as pd
import numpy as np
import os

pd.set_option("display.width", 120)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"Loaded {len(df)} rows, {df['client_id'].nunique()} clients")

# NOTE: this label is used ONLY to audit whether the two signals below hold in the data.
# It is never used as an input to the rule/score built in Section 2.
df["is_declining_label"] = (df["trend_direction"].str.lower() == "down").astype(int)
base_rate = df["is_declining_label"].mean()
print(f"\nBase decline rate across all {len(df)} rows: {base_rate:.3f}")

# --- Signal check 1: staleness (behind refresh flags) ---
bins = [0, 14, 30, 90, 180, 400]
labels = ["0-14", "15-30", "31-90", "91-180", "181+"]
df["staleness_bucket"] = pd.cut(df["days_since_last_update"], bins=bins, labels=labels)

staleness_table = (
    df.groupby("staleness_bucket", observed=True)["is_declining_label"]
    .agg(n="count", decline_rate="mean")
)
print("\n=== Signal 1: staleness (days_since_last_update) vs decline rate ===")
print(staleness_table)
print("Verdict: MIXED")

# --- Signal check 2: visibility / volume (behind quick-win logic) ---
volume_table = (
    df.groupby("impression_tier", observed=True)["is_declining_label"]
    .agg(n="count", decline_rate="mean")
    .reindex(["low", "moderate", "good", "excellent"])
)
print("\n=== Signal 2: search visibility (impression_tier) vs decline rate ===")
print(volume_table)
print("Verdict: MIXED")

Loaded 30000 rows, 32 clients

Base decline rate across all 30000 rows: 0.542

=== Signal 1: staleness (days_since_last_update) vs decline rate ===
                      n  decline_rate
staleness_bucket                     
0-14               3933      0.510043
15-30             16547      0.511694
31-90               175      0.588571
91-180             9171      0.611057
181+                174      0.471264
Verdict: MIXED

=== Signal 2: search visibility (impression_tier) vs decline rate ===
                     n  decline_rate
impression_tier                     
low              11248      0.453947
moderate         10469      0.614672
good              7205      0.586121
excellent         1078      0.461967
Verdict: MIXED


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

### Ranked queue

The baseline ranks pages by the number of recent impressions, but only after they satisfy both conditions of the rule: at least 180 days since the last update and at least 500 impressions in the recent 90-day window.

The queue is intended to support human review. A high score means higher review priority, not that a refresh is guaranteed to improve performance.

In [10]:
STALE_DAYS = 90
VISIBLE_IMPRESSIONS = 300

df["stale"] = (df["days_since_last_update"] >= STALE_DAYS).astype(int)
df["visible"] = (df["impressions_90d"] >= VISIBLE_IMPRESSIONS).astype(int)

df["baseline_action_score"] = df["stale"] * df["visible"] * df["impressions_90d"]

flagged = (df["stale"] == 1) & (df["visible"] == 1)
df["reason_code"] = np.where(flagged, "stale_visible_page", "not_flagged")
df["action_label"] = np.where(flagged, "review_for_refresh", "monitor")

df["rank"] = df["baseline_action_score"].rank(method="first", ascending=False).astype(int)

output_columns = [
    "content_id", "client_id", "rank", "baseline_action_score",
    "reason_code", "action_label",
    "days_since_last_update", "impressions_90d", "avg_position", "ctr",
    "content_age_days", "word_count", "content_type",
]

queue = df[output_columns].sort_values("rank").reset_index(drop=True)

os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)

print(f"Wrote {len(queue)} rows to work/outputs/baseline_action_score.csv")
print(f"Flagged review_for_refresh: {flagged.sum()} of {len(df)} ({flagged.mean():.1%})")
print()
print(queue.head(10).to_string())


Wrote 30000 rows to work/outputs/baseline_action_score.csv
Flagged review_for_refresh: 7234 of 30000 (24.1%)

             content_id          client_id  rank  baseline_action_score         reason_code        action_label  days_since_last_update  impressions_90d  avg_position   ctr  content_age_days  word_count     content_type
0  content_5fe46e04994d  client_4e07408562     1                 517715  stale_visible_page  review_for_refresh                     104           517715           4.2  0.14               537         NaN  keyword article
1  content_2dba2b1f9536  client_6208ef0f77     2                 443434  stale_visible_page  review_for_refresh                     104           443434          27.9  0.21               299      7676.0  keyword article
2  content_2c2606c5d176  client_19581e27de     3                 347399  stale_visible_page  review_for_refresh                     104           347399           4.2  0.53               362         NaN  keyword article
3  content

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

For each of the top ten: the action, why it's there, and what would make it wrong.

1. **content_5fe46e04994d** (rank 1) — `review_for_refresh`. Why: the single highest-visibility stale
   page in the dataset (517,715 impressions/90d), already declining (`trend=down`), position 4.2. What
   would make it wrong: `word_count` is missing, so I can't confirm this is actually a rewritable article
   rather than a thin/aggregated page; also CTR is only 0.14% at position 4.2 — that's low enough that a
   CTR/snippet fix might matter more than a content refresh.

2. **content_2dba2b1f9536** (rank 2) — `review_for_refresh`. Why: second-largest audience among stale
   pages, 7,676-word article stuck at position 27.9 with low CTR — plausible refresh-to-improve-rank case.
   What would make it wrong: `trend=stable`, not declining — this page isn't currently losing anything, so
   it may be lower urgency than pages that are actively sliding.

3. **content_2c2606c5d176** (rank 3) — `review_for_refresh`. Why: position 4.2, high visibility,
   declining. What would make it wrong: same missing-`word_count` blind spot as #1; CTR here (0.53%) is
   closer to a plausible value for that position, so this one looks like a cleaner "genuine refresh"
   candidate than #1.

4. **content_cb112fce36be** (rank 4) — `review_for_refresh`. Why: strong position (5.6), big audience,
   already declining, and — unlike most of this list — actually young (126 days old) yet already stale by
   update history. What would make it wrong: if the underlying keyword's search demand itself is fading
   (a category-level trend, not a content problem), no refresh fixes that — worth checking `search_volume`
   for the target keyword before committing review time.

5. **content_9532f197bbc8** (rank 5) — `review_for_refresh`. Why: position 2.0 — effectively the top of
   the page — with a huge audience and `trend=down`. What would make it wrong: CTR is only 0.87% at
   position 2, far below what a position-2 result should draw. That smells like a CTR/meta-description
   problem, not a "needs more content" problem — my rule's single reason code can't tell these apart, and
   routing this to a content refresh may waste review time on the wrong fix.

6. **content_36ff89c8214e** (rank 6) — `review_for_refresh`. Why: big audience, stale. What would make it
   wrong: CTR is 0.05% at position 7.3 — unusually low even for that position, closer to a possible
   indexing/snippet/tracking issue than a content-quality one; also `trend=stable`, so urgency may be
   overstated.

7. **content_b28d1efd668f** (rank 7) — `review_for_refresh`. Why: 6,901-word article, big audience,
   stuck at position 26.2. What would make it wrong: `trend=stable` — not urgent — and if position 26.2
   reflects real keyword competitiveness rather than thin content, a refresh alone won't move it enough to
   justify the review time.

8. **content_813e88069237** (rank 8) — `review_for_refresh`. Why: same client and same position (26.2)
   as #7, but this one is `trend=down` — actively losing ground while #7 is merely flat. Arguably this
   should outrank #7, which shows the score's "audience size only" tiebreak misses trend direction.
   What would make it wrong: if #7 and #8 target overlapping keywords for the same client, fixing one
   might already address both.

9. **content_c21024970297** (rank 9) — `review_for_refresh`. Why: decent position (5.1) with the most
   "normal-looking" CTR (0.41%) of the top ten relative to its rank — the least obviously broken pick, and
   probably the safest bet that a refresh (rather than a CTR fix) is the right lever. What would make it
   wrong: `trend=stable` — this page may already be near its ceiling, so review time here is closer to
   "protect what's working" than "fix a problem."

10. **content_c8e9d6ab9013** (rank 10) — `review_for_refresh`. Why: ~200k impressions but a **0.00% CTR**
    — literally zero recorded clicks over 90 days at position 9.7. What would make it wrong: a true 0%
    CTR at that volume is unusual enough that it could be a measurement gap (GSC data issue, bot-inflated
    impressions) rather than a real content problem — this is exactly the kind of row a human should sanity
    check before spending review time on it.


In [11]:
pd.set_option("display.max_columns", 20)
print(queue.head(10).to_string())



             content_id          client_id  rank  baseline_action_score         reason_code        action_label  days_since_last_update  impressions_90d  avg_position   ctr  content_age_days  word_count     content_type
0  content_5fe46e04994d  client_4e07408562     1                 517715  stale_visible_page  review_for_refresh                     104           517715           4.2  0.14               537         NaN  keyword article
1  content_2dba2b1f9536  client_6208ef0f77     2                 443434  stale_visible_page  review_for_refresh                     104           443434          27.9  0.21               299      7676.0  keyword article
2  content_2c2606c5d176  client_19581e27de     3                 347399  stale_visible_page  review_for_refresh                     104           347399           4.2  0.53               362         NaN  keyword article
3  content_cb112fce36be  client_19581e27de     4                 309910  stale_visible_page  review_for_refresh         

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak picks worth flagging to a reviewer:**

- **The 104-day cluster.** 8,773 rows (29% of the dataset) share `days_since_last_update == 104` exactly
  — almost certainly a bulk-update date (CMS migration or import), not organic staleness. Every row in the
  top 10 ties on that exact value, so within this cohort the ranking is really just "biggest audience,"
  and the `stale` flag isn't distinguishing *how* stale a page is, only whether it's in that batch or not.
- **`trend=stable` pages in the queue (#2, #6, #7, #9).** The rule flags pages worth revisiting, not only
  pages currently failing. That's a defensible design (it's forward-looking, not just reactive) but a
  content team expecting an "actively declining" list should know the queue includes flat performers too.
- **Missing `word_count` in 5 of the top 10.** Nearly half the front of the queue can't be checked for
  content depth from this data alone — a real blind spot for anyone trying to act on rank 1 first.
- **Suspiciously low/zero CTR at good positions (#1, #5, #6, #10).** These look more like CTR/snippet
  problems than content-depth problems. A single reason code can't separate "needs a rewrite" from "needs
  a meta-description fix" — a known limitation of collapsing this into one rule and one code.

**Leakage check** — confirm the score used only the two intended columns, and nothing label-derived or
future-windowed:


In [12]:
inputs_used = {"days_since_last_update", "impressions_90d"}

forbidden = {
    "trend_direction", "trend_pct", "is_declining_label",
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
}

print("Columns the score/reason_code/action_label were built from:", inputs_used)
print("Overlap with forbidden future-window / label-derived columns:", inputs_used & forbidden)
print("Clean:", inputs_used.isdisjoint(forbidden))

# Sanity check: none of the CSV's output columns are label-derived or future-window either.
csv_cols = set(pd.read_csv("work/outputs/baseline_action_score.csv").columns)
print("\nOutput CSV columns:", sorted(csv_cols))
print("Any forbidden columns leaked into the CSV:", bool(csv_cols & forbidden))



Columns the score/reason_code/action_label were built from: {'days_since_last_update', 'impressions_90d'}
Overlap with forbidden future-window / label-derived columns: set()
Clean: True

Output CSV columns: ['action_label', 'avg_position', 'baseline_action_score', 'client_id', 'content_age_days', 'content_id', 'content_type', 'ctr', 'days_since_last_update', 'impressions_90d', 'rank', 'reason_code', 'word_count']
Any forbidden columns leaked into the CSV: False


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.